In [2]:
# =============================================================================
# Autor:   Thiago Souza (@engthiago1979-blip)
# Criado:  19-04-2026
# Uso:     Estatística avançada (MK, KW, Descritiva), P95 e plots para todo o escopo CONAMA 430.
# =============================================================================

import os
import pandas as pd
import numpy as np
import scipy.stats as stats
import pymannkendall as mk
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings

# Ignora warnings de divisão por zero do numpy/scipy em séries sem variância
warnings.filterwarnings("ignore")

# --- 1. CONFIGURAÇÕES ---
DIR_BASE = r"C:\Users\thiago\Documents\estudos_python\etes_nesa\base_dados"
DIR_PRODUTOS = r"C:\Users\thiago\Documents\estudos_python\etes_nesa\Efluentes_Sanitários\produtos"
FILE_INPUT = os.path.join(DIR_BASE, "ETEs_NorteEnergia_Processado_v1.xlsx")

os.makedirs(DIR_PRODUTOS, exist_ok=True)

NESA_COLORS = {'primary': '#009CA7', 'vmp': '#F25A3C', 'text': '#404041', 'highlight': '#B9F8FF'}
plt.rcParams['font.family'] = 'Segoe UI'

# --- 2. VMPs CONAMA 430/11 (Art. 16) ---
VMP_CONAMA = {
    'DEMANDA_BIOQUIMICA_DE_OXIGENIO': 120.0,
    'PH': (5.0, 9.0),
    'TEMPERATURA_DO_EFLUENTE': 40.0,
    'MATERIAIS_SEDIMENTAVEIS': 1.0,
    'OLEOS_E_GRAXAS': 50.0,
    'ARSENIO_TOTAL': 0.5,
    'CADMIO_TOTAL': 0.2,
    'CHUMBO_TOTAL': 0.5,
    'CIANETO_TOTAL': 1.0,
    'CROMO_HEXAVALENTE': 0.1,
    'CROMO_TRIVALENTE': 1.0,
    'FERRO_DISSOLVIDO': 15.0,
    'MANGANES_DISSOLVIDO': 1.0,
    'MERCURIO_TOTAL': 0.01,
    'SULFETO_TOTAL': 1.0,
    'BENZENO': 1.2,
    'CLOROFORMIO': 1.0,
    'ETILBENZENO': 1.4,
    'FENOIS_TOTAIS': 0.5,
    'TETRACLORETO_DE_CARBONO': 1.0,
    'TRICLOROETENO': 1.0,
    'TOLUENO': 1.2,
    'XILENO_TOTAL': 1.6
}

# --- 3. MOTORES ESTATÍSTICOS ---
def analisar_parametro(df, parametro, limite_conama, col_ete):
    """Calcula Descritiva, Auditoria P95 e Tendência de Mann-Kendall."""
    resultados = []
    
    for ete in df[col_ete].dropna().unique():
        df_ete = df[df[col_ete] == ete].sort_values('DATA')
        serie = df_ete[parametro].dropna()
        
        if len(serie) < 3: 
            continue
            
        # 3.1 Estatística Descritiva Básica
        media = serie.mean()
        desvio = serie.std()
        minimo = serie.min()
        maximo = serie.max()
        p95 = np.percentile(serie, 95)
        
        # 3.2 Conformidade e P95
        if isinstance(limite_conama, tuple):
            vmin, vmax = limite_conama
            amostras_ok = len(serie[(serie >= vmin) & (serie <= vmax)])
            status_auditoria = "ROBUSTO" if (serie.min() >= vmin and p95 <= vmax) else "RISCO"
            limite_str = f"{vmin} a {vmax}"
        else:
            amostras_ok = len(serie[serie <= limite_conama])
            status_auditoria = "ROBUSTO" if p95 <= limite_conama else "RISCO"
            limite_str = str(limite_conama)
            
        taxa_conformidade = (amostras_ok / len(serie)) * 100
        
        # 3.3 Teste de Tendência (Mann-Kendall)
        # Proteção contra série sem variância (ex: só valores <LQ)
        if desvio > 0 and len(serie) >= 4:
            try:
                mk_res = mk.original_test(serie)
                tendencia = mk_res.trend # 'increasing', 'decreasing', 'no trend'
                p_val_mk = mk_res.p
                # Tradução
                if tendencia == 'increasing': tendencia = 'Crescente (Piora)'
                elif tendencia == 'decreasing': tendencia = 'Decrescente (Melhora)'
                else: tendencia = 'Estável'
            except:
                tendencia, p_val_mk = "Erro Cálculo", np.nan
        else:
            tendencia, p_val_mk = "Sem variância/dados", np.nan

        resultados.append({
            'Parâmetro': parametro.replace('_', ' '),
            'ETE': ete, 
            'N': len(serie),
            'Mínimo': round(minimo, 3),
            'Média': round(media, 3),
            'Máximo': round(maximo, 3),
            'Desvio Padrão': round(desvio, 3),
            'P95 (Auditoria)': round(p95, 3),
            'VMP CONAMA': limite_str,
            'Conformidade (%)': f"{taxa_conformidade:.1f}%",
            'Avaliação': status_auditoria,
            'Tendência (MK)': tendencia,
            'p-value (MK)': round(p_val_mk, 4) if pd.notna(p_val_mk) else "N/A"
        })
        
    return resultados

def teste_kruskal_wallis_geral(df, parametro, col_ete):
    """Verifica se há diferença significativa de desempenho entre as ETEs."""
    grupos = []
    nomes_etes = []
    
    for ete in df[col_ete].dropna().unique():
        serie = df[df[col_ete] == ete][parametro].dropna()
        if len(serie) >= 3 and serie.std() > 0:
            grupos.append(serie)
            nomes_etes.append(ete)
            
    if len(grupos) > 1:
        stat, p_val = stats.kruskal(*grupos)
        significativo = "SIM" if p_val < 0.05 else "NÃO"
        return {
            'Parâmetro': parametro.replace('_', ' '),
            'ETEs Comparadas': " vs ".join(nomes_etes),
            'Estatística (H)': round(stat, 3),
            'p-value (KW)': round(p_val, 4),
            'Diferença Significativa?': significativo
        }
    return None

def plotar_matplotlib_v4(df, parametro, ete, col_ete, dir_out):
    """Plot estático de Alta Resolução (Matplotlib)."""
    df_plot = df[df[col_ete] == ete].sort_values('DATA').dropna(subset=[parametro])
    if len(df_plot) < 2: return
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df_plot['DATA'], df_plot[parametro], marker='o', linestyle='-', 
            color=NESA_COLORS['primary'], linewidth=2)
    
    limite = VMP_CONAMA[parametro]
    if isinstance(limite, tuple):
        ax.axhline(y=limite[0], color=NESA_COLORS['vmp'], linestyle='--', linewidth=1.5)
        ax.axhline(y=limite[1], color=NESA_COLORS['vmp'], linestyle='--', linewidth=1.5, label=f"CONAMA ({limite[0]} a {limite[1]})")
    else:
        ax.axhline(y=limite, color=NESA_COLORS['vmp'], linestyle='--', linewidth=1.5, label=f"VMP ({limite})")
    
    # Banda de P95 para auditoria visual
    p95 = np.percentile(df_plot[parametro], 95)
    ax.axhline(y=p95, color=NESA_COLORS['highlight'], linestyle='-.', linewidth=2, label=f"P95 ({p95:.2f})")
    
    ax.set_title(f"{parametro.replace('_', ' ')}\nETE: {ete}", fontsize=12, color=NESA_COLORS['text'], loc='left')
    ax.set_ylabel("Concentração (mg/L)", fontsize=10)
    
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
    plt.xticks(rotation=45)
    ax.grid(True, axis='y', linestyle=':', alpha=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(frameon=False, loc='best', fontsize=9)
    
    nome_arquivo = f"Plot_{parametro}_{ete.replace(' ', '_')}.png"
    plt.savefig(os.path.join(dir_out, nome_arquivo), dpi=600, bbox_inches='tight')
    plt.close()

# --- 4. EXECUÇÃO DO PIPELINE ---
if __name__ == "__main__":
    try:
        df = pd.read_excel(FILE_INPUT)
        col_ete = [c for c in df.columns if 'DESCRICAO' in c.upper() and 'PONTO' in c.upper()][0]

        # Correção de ruídos (Obrigatório)
        df[col_ete] = df[col_ete].astype(str).str.upper().str.strip()
        df[col_ete] = df[col_ete].replace({'ETE 1': 'ETE 01', 'ETE 2': 'ETE 02'})

        relatorio_geral = []
        relatorio_kw = []

        print("[*] Iniciando processamento estatístico completo...")
        
        for param in VMP_CONAMA.keys():
            if param in df.columns:
                print(f"    -> Analisando: {param}")
                
                # 1. Estatística Básica, MK e P95
                resultados_descritiva = analisar_parametro(df, param, VMP_CONAMA[param], col_ete)
                if resultados_descritiva:
                    relatorio_geral.extend(resultados_descritiva)
                
                # 2. Teste de Hipótese (Kruskal-Wallis)
                resultado_kw = teste_kruskal_wallis_geral(df, param, col_ete)
                if resultado_kw:
                    relatorio_kw.append(resultado_kw)
                
                # 3. Geração Gráfica
                for ete in df[col_ete].unique():
                    plotar_matplotlib_v4(df, param, ete, col_ete, DIR_PRODUTOS)

        # Exportação Multi-Aba
        df_final = pd.DataFrame(relatorio_geral)
        df_kw = pd.DataFrame(relatorio_kw)
        
        caminho_excel = os.path.join(DIR_PRODUTOS, "Relatorio_Analitico_Mestre.xlsx")
        with pd.ExcelWriter(caminho_excel, engine='openpyxl') as writer:
            df_final.to_excel(writer, sheet_name="Estatistica_e_Tendencia", index=False)
            if not df_kw.empty:
                df_kw.to_excel(writer, sheet_name="Comparativo_Kruskal_Wallis", index=False)

        print(f"\n[+] SUCESSO! Relatório Mestre com todas as estatísticas salvo em: {caminho_excel}")
        
    except Exception as e:
        print(f"[!] Erro crítico: {e}")

[*] Iniciando processamento estatístico completo...
    -> Analisando: DEMANDA_BIOQUIMICA_DE_OXIGENIO
    -> Analisando: PH
    -> Analisando: TEMPERATURA_DO_EFLUENTE
    -> Analisando: MATERIAIS_SEDIMENTAVEIS
    -> Analisando: OLEOS_E_GRAXAS
    -> Analisando: ARSENIO_TOTAL
    -> Analisando: CADMIO_TOTAL
    -> Analisando: CHUMBO_TOTAL
    -> Analisando: CIANETO_TOTAL
    -> Analisando: CROMO_HEXAVALENTE
    -> Analisando: CROMO_TRIVALENTE
    -> Analisando: FERRO_DISSOLVIDO
    -> Analisando: MANGANES_DISSOLVIDO
    -> Analisando: MERCURIO_TOTAL
    -> Analisando: SULFETO_TOTAL
    -> Analisando: BENZENO
    -> Analisando: CLOROFORMIO
    -> Analisando: ETILBENZENO
    -> Analisando: FENOIS_TOTAIS
    -> Analisando: TETRACLORETO_DE_CARBONO
    -> Analisando: TRICLOROETENO
    -> Analisando: TOLUENO
    -> Analisando: XILENO_TOTAL

[+] SUCESSO! Relatório Mestre com todas as estatísticas salvo em: C:\Users\thiago\Documents\estudos_python\etes_nesa\Efluentes_Sanitários\produtos\Relato

In [3]:
# =============================================================================
# Autor:   Thiago Souza (@engthiago1979-blip)
# Criado:  19-04-2026
# Uso:     Módulo QA/QC: Gera tabela e mapa de calor quantificando os dados 
#          censurados (<LQ) por Parâmetro e por ETE.
# =============================================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. CONFIGURAÇÕES ---
DIR_BASE = r"C:\Users\thiago\Documents\estudos_python\etes_nesa\base_dados"
DIR_PRODUTOS = r"C:\Users\thiago\Documents\estudos_python\etes_nesa\Efluentes_Sanitários\produtos"
FILE_INPUT = os.path.join(DIR_BASE, "ETEs_NorteEnergia_Processado_v1.xlsx")

NESA_COLORS = {'primary': '#009CA7', 'text': '#404041'}
plt.rcParams['font.family'] = 'Segoe UI'

# --- 2. EXECUÇÃO ---
try:
    print("[*] Lendo a base de dados processada para auditoria de QA/QC...")
    df = pd.read_excel(FILE_INPUT)
    
    # Identificar a coluna da ETE dinamicamente e corrigir ruídos
    col_ete = [c for c in df.columns if 'DESCRICAO' in c.upper() and 'PONTO' in c.upper()][0]
    df[col_ete] = df[col_ete].astype(str).str.upper().str.strip()
    df[col_ete] = df[col_ete].replace({'ETE 1': 'ETE 01', 'ETE 2': 'ETE 02'})

    # Separar as colunas (Metadados vs Parâmetros analíticos)
    colunas_metadados = df.columns[:4].tolist()
    parametros = [c for c in df.columns if c not in colunas_metadados and not c.endswith('_IMPUTADO_LQ') and not c.endswith('_STATUS_VMP')]

    dados_qc = []

    print("[*] Contabilizando matriz de dados censurados...")
    for param in parametros:
        col_flag = param + "_IMPUTADO_LQ"
        has_flag = col_flag in df.columns

        for ete in df[col_ete].unique():
            df_ete = df[df[col_ete] == ete]
            serie_param = df_ete[param].dropna()
            n_total = len(serie_param)

            if n_total > 0:
                if has_flag:
                    # Conta os 'True' na coluna flag, apenas onde o parâmetro possui dado preenchido
                    n_censurado = df_ete.loc[serie_param.index, col_flag].fillna(False).sum()
                else:
                    n_censurado = 0

                perc_censurado = (n_censurado / n_total) * 100

                dados_qc.append({
                    'ETE': ete,
                    'Parâmetro': param.replace('_', ' '),
                    'Total de Amostras (N)': int(n_total),
                    'Dados Censurados (<LQ)': int(n_censurado),
                    '% Censurado': round(perc_censurado, 1)
                })

    # Criando o DataFrame Final
    df_qc = pd.DataFrame(dados_qc)
    
    # 1. Exportando a Tabela
    caminho_excel = os.path.join(DIR_PRODUTOS, "Relatorio_QAQC_Dados_Censurados.xlsx")
    df_qc.to_excel(caminho_excel, index=False)
    
    # 2. Gerando o Mapa de Calor (Heatmap)
    # Filtramos apenas os parâmetros que tiveram dados censurados para o gráfico não ficar gigante e vazio
    df_qc_filtrado = df_qc[df_qc['% Censurado'] > 0]
    
    if not df_qc_filtrado.empty:
        pivot_qc = df_qc_filtrado.pivot(index='Parâmetro', columns='ETE', values='% Censurado').fillna(0)
        
        plt.figure(figsize=(10, max(6, len(pivot_qc) * 0.4))) # Altura dinâmica baseada na qtd de parâmetros
        ax = sns.heatmap(pivot_qc, annot=True, fmt=".1f", cmap="GnBu", linewidths=1, 
                         cbar_kws={'label': '% de Amostras Censuradas (<LQ)'})
        
        plt.title("Mapa de Calor (QA/QC): Incidência de Dados Censurados por ETE", 
                  fontsize=14, color=NESA_COLORS['text'], pad=20)
        plt.ylabel("Parâmetro Analítico", fontsize=11, fontweight='bold', color=NESA_COLORS['text'])
        plt.xlabel("Estação de Tratamento (ETE)", fontsize=11, fontweight='bold', color=NESA_COLORS['text'])
        
        caminho_fig = os.path.join(DIR_PRODUTOS, "Heatmap_QAQC_Censurados.png")
        plt.savefig(caminho_fig, dpi=300, bbox_inches='tight')
        plt.close()

    print(f"\n[+] SUCESSO! Produtos gerados em: {DIR_PRODUTOS}")
    print("  -> Tabela: Relatorio_QAQC_Dados_Censurados.xlsx")
    if not df_qc_filtrado.empty:
        print("  -> Gráfico: Heatmap_QAQC_Censurados.png")

    print("\n=== TOP 10 MAIORES INCIDÊNCIAS DE CENSURA (<LQ) ===")
    top_10 = df_qc.sort_values(by='% Censurado', ascending=False).head(10)
    print(top_10.to_string(index=False))

except Exception as e:
    print(f"[!] Erro no processamento de QA/QC: {e}")

[*] Lendo a base de dados processada para auditoria de QA/QC...
[*] Contabilizando matriz de dados censurados...

[+] SUCESSO! Produtos gerados em: C:\Users\thiago\Documents\estudos_python\etes_nesa\Efluentes_Sanitários\produtos
  -> Tabela: Relatorio_QAQC_Dados_Censurados.xlsx
  -> Gráfico: Heatmap_QAQC_Censurados.png

=== TOP 10 MAIORES INCIDÊNCIAS DE CENSURA (<LQ) ===
         ETE                                Parâmetro  Total de Amostras (N)  Dados Censurados (<LQ)  % Censurado
      ETE 01                            ARSENIO TOTAL                     23                      23        100.0
      ETE PM                            ARSENIO TOTAL                     18                      18        100.0
ETE COMPACTA                         CROMO TRIVALENTE                     21                      21        100.0
      ETE PM                         CROMO TRIVALENTE                     18                      18        100.0
      ETE 01 DICLOROETENO SOMATORIA DE 1112CIS12TRANS   